# CFO Finance Lab — take-home notebook

Hands-on companion to **Act 9** of the DataAI_Amity guest lecture. Five stages, one CFO storyline:

**MESS → TRUST → DECIDE → PREDICT → ASK**

Everything runs locally against the FastAPI backend that ships with the lecture app. Default LLM is **Ollama** with `gemma4:latest` (or `llama3:latest`). Swap providers via `.env`.

## How to run

1. Start the backend: `cd app/backend && uvicorn main:app --reload --port 8000`
2. (For the ASK stage) Start Ollama: `ollama serve` and `ollama pull gemma4:latest`
3. Run cells top-to-bottom.

In [ ]:
import requests, pandas as pd
BASE = 'http://localhost:8000/api/finance'

def get(path, **params):
    r = requests.get(f'{BASE}{path}', params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def post(path, **body):
    r = requests.post(f'{BASE}{path}', json=body, timeout=60)
    r.raise_for_status()
    return r.json()

# Make sure the warehouse is built. Idempotent.
post('/reseed')
post('/run')
get('/counts')

## Stage 1 — MESS

Two raw source files, two different shapes. Same vendor, different strings.

In [ ]:
concur = pd.DataFrame(get('/raw/concur', limit=50)['rows'])
card = pd.DataFrame(get('/raw/card', limit=50)['rows'])

print('Concur columns:', list(concur.columns))
print('Card columns:  ', list(card.columns))
concur.head(5)

In [ ]:
card.head(5)

Notice:

- Concur has multiple date formats and a `currency` column (AED + USD).
- Card is USD-only, has cryptic vendor strings (`AMZN MKTPLACE`, `UBER *TRIP`), and `employee_id` is sometimes NULL.

Neither is wrong. They're just **different**, which is the natural state of business data.

## Stage 2 — TRUST

Run the Bronze → Silver → Gold pipeline. Watch counts and quality flags.

In [ ]:
result = post('/run')
result

In [ ]:
silver = pd.DataFrame(get('/sample/silver', limit=200)['rows'])
print(f'Silver rows shown: {len(silver)} (full table is /counts.silver)')
silver.head(8)

In [ ]:
# Same vendor, both sources, after canonicalization.
amazon = silver[silver['vendor'] == 'Amazon']
print(f"{len(amazon)} Amazon rows in Silver, sources: {amazon['source'].value_counts().to_dict()}")
amazon.head(5)

## Stage 3 — DECIDE

Two Gold marts drive the BI dashboards. Both are aggregates over Silver.

In [ ]:
kpis = get('/marts/kpis')
kpis

In [ ]:
spend = pd.DataFrame(get('/marts/spend-by-dept-month')['rows'])
pivot = spend.pivot(index='month_label', columns='dept_name', values='total_aed').fillna(0)
pivot.tail(6)

In [ ]:
import matplotlib.pyplot as plt
ax = pivot.plot(kind='bar', stacked=True, figsize=(9, 4))
ax.set_title('Spend by department × month (AED)')
ax.set_xlabel('')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()

In [ ]:
vendors = pd.DataFrame(get('/marts/top-vendors', limit=15)['rows'])
vendors[['canonical_name', 'category', 'total_aed', 'pareto_rank', 'cumulative_pct']].head(15)

Drill from a Gold cell back into the Silver row-level detail — same idea as a click-through in Power BI.

In [ ]:
# Pick the most recent Marketing month from the spend mart.
marketing_rows = spend[spend['dept_name'] == 'Marketing'].sort_values(['year', 'month']).tail(1)
row = marketing_rows.iloc[0]
drill = get('/marts/drill/dept', dept_id=row['dept_id'], year=int(row['year']), month=int(row['month']))
pd.DataFrame(drill['rows']).head(8)

## Stage 4 — PREDICT

Isolation Forest over (amount, day-of-week, dept, category). Sensitivity is the contamination parameter — what fraction of rows we expect to be outliers.

In [ ]:
anom = get('/anomalies', sensitivity=0.05, top_k=10)
print(f"Total: {anom['total_rows']}, flagged: {anom['outlier_count']}")
pd.DataFrame(anom['flagged'])[['txn_date', 'employee', 'department', 'vendor', 'category', 'amount_aed', 'reasons']]

In [ ]:
# Try a wider net (more sensitive) to see false positives appear.
loose = get('/anomalies', sensitivity=0.15, top_k=10)
print(f"Sensitivity 15%: flagged={loose['outlier_count']} (vs 5% above)")
pd.DataFrame(loose['flagged'])[['txn_date', 'department', 'vendor', 'amount_aed', 'reasons']].head(10)

## Stage 5 — ASK (text-to-SQL)

The model writes SQL, the backend runs it against the Gold mart, you get rows. Default provider is **Ollama**. If your `.env` points at OpenRouter / Azure / NVIDIA, this still works without code changes.

In [ ]:
health = get('/llm/health')
health

In [ ]:
# Warm the model so the first real call isn't slow.
post('/llm/warmup')

In [ ]:
presets = get('/ask/presets')['presets']
presets

In [ ]:
answer = post('/ask', question='Top 5 vendors by total spend.')
print('SQL the model wrote:\n')
print(answer.get('sql', '(no sql)'))
print()
if answer.get('rows'):
    display(pd.DataFrame(answer['rows']))
else:
    print('error:', answer.get('error'))

In [ ]:
# Your turn — type any question.
your_question = 'Show me Amazon spend by month.'
answer = post('/ask', question=your_question)
print(answer.get('sql', ''))
if answer.get('rows'):
    display(pd.DataFrame(answer['rows']))
else:
    print('error:', answer.get('error'))

## Where to go next

- Open Act 9 in the React app at <http://localhost:8000/act/9> for the visual version.
- Edit `app/backend/finance/text_to_sql.py` to add few-shot examples for your own domain.
- Edit `.env` to point at OpenRouter / Azure OpenAI / NVIDIA NIM — the same code answers your questions through any of them.
- Add new departments / vendors / categories in `app/backend/finance/seed_finance.py` and watch the dashboards adapt.